# Transcripción y diarización de audio

Pipeline de [`scripts/test_audio_ovh.py`](../scripts/test_audio_ovh.py) paso a paso:

**vídeo → audio → Whisper en OVHcloud (con diarización) → turnos de habla → CSV del contrato → tiempo de habla por hablante**

## 0. Configuración

In [30]:
import os, json, subprocess
from pathlib import Path
import requests
import pandas as pd

VIDEO = "videos/robot_social_CSIC.mp4"   # <- vídeo a analizar
LANG = "es"                                                       # idioma del audio: ca, es, en...

BASE = os.environ.get("OVH_BASE_URL", "https://oai.endpoints.kepler.ai.cloud.ovh.net/v1")
TOKEN = "".join((os.environ.get("OVH_AI_ENDPOINTS_ACCESS_TOKEN") or os.environ.get("OVH_API_TOKEN") or "").split())
ASR_MODEL = os.environ.get("OVH_ASR_MODEL", "whisper-large-v3")
assert TOKEN, "Falta el token: define la variable de entorno OVH_AI_ENDPOINTS_ACCESS_TOKEN"

## 1. Extraer el audio del vídeo

WAV mono a 16 kHz: es el formato con el que trabaja Whisper y pesa menos al subirlo.

In [31]:
AUDIO = Path(VIDEO).with_suffix(".wav").name
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", VIDEO, "-vn", "-ac", "1", "-ar", "16000", AUDIO], check=True)
print(f"{AUDIO}: {Path(AUDIO).stat().st_size / 1e6:.1f} MB")

robot_social_CSIC.wav: 4.8 MB


## 2. Transcribir con Whisper en OVHcloud

`diarize=true` pide que, además del texto, se indique qué hablante dice cada fragmento. Guardamos la respuesta cruda en JSON para poder revisarla.

In [32]:
with open(AUDIO, "rb") as f:
    r = requests.post(f"{BASE}/audio/transcriptions",
                      headers={"Authorization": f"Bearer {TOKEN}"},
                      files={"file": (AUDIO, f)},
                      data={"model": ASR_MODEL, "language": LANG, "response_format": "verbose_json", "diarize": "true"},
                      timeout=180)
assert r.ok, f"Error {r.status_code}: {r.text[:500]}"
res = r.json()

Path(AUDIO).with_suffix(".json").write_text(json.dumps(res, ensure_ascii=False, indent=2), encoding="utf-8")
list(res.keys())

['task',
 'success',
 'language',
 'duration',
 'text',
 'words',
 'segments',
 'diarization',
 'usage']

## 3. Turnos de habla

OVHcloud devuelve los turnos con su hablante en la clave `diarization` (si no está, usamos `segments`, sin hablante).

In [39]:
turnos = res.get("diarization") or res.get("segments", [])

df = pd.DataFrame([{
    "start": float(s.get("start", 0.0)),
    "end": float(s.get("end", 0.0)),
    "speaker": f"Hablante_{s['speaker']}" if isinstance(s.get("speaker"), int) else str(s.get("speaker", "Desconocido")),
    "text": s.get("text", "").strip(),
} for s in turnos])
print(len(df), "turnos")
df.head(30)

22 turnos


,start,end,speaker,text
0,0.00,1.60,Hablante_0,"Hey Pepe, mírame."
1,1.92,3.44,Hablante_0,¿De qué color es mi cara?
2,3.68,7.10,Hablante_0,El color... ¡verde!
3,7.90,9.20,Hablante_0,"Muy bien, Pepe."
4,10.04,18.98,Hablante_1,En España hay unos 100.000 niños y niñas con t...
5,18.98,34.26,Hablante_1,Desde CESIC y con el apoyo de la serie estamos...
6,34.92,37.28,Hablante_0,Une las caras con la misma emoción.
7,37.90,38.52,Hablante_0,Triste.
8,39.74,40.36,Hablante_0,Sorprendido.
9,40.74,41.88,Hablante_0,Súper Pepe.


## 4. Género de la voz

Clasificador [`prithivMLmods/Common-Voice-Gender-Detection`](https://huggingface.co/prithivMLmods/Common-Voice-Gender-Detection) (código adaptado de su documentación). Recortamos el audio de cada turno con sus tiempos de inicio y fin, lo clasificamos y decidimos una etiqueta por hablante con la media de sus turnos.

In [34]:
import torch
from scipy.io import wavfile
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

# Load model and processor
model_name = "prithivMLmods/Common-Voice-Gender-Detection"
model = Wav2Vec2ForSequenceClassification.from_pretrained(model_name)
processor = Wav2Vec2FeatureExtractor.from_pretrained(model_name)

# Label mapping
id2label = {"0": "female", "1": "male"}

# Audio completo del paso 1: ya es mono a 16 kHz, lo que espera el modelo
SR = 16000
_, audio = wavfile.read(AUDIO)
audio = audio.astype("float32") / 32768          # de enteros de 16 bits a decimales entre -1 y 1

def classify_audio(speech):
    """Como en la documentación, pero recibe un trozo de audio (array a 16 kHz) en vez de la ruta a un archivo."""
    inputs = processor(speech, sampling_rate=SR, return_tensors="pt", padding=True)
    with torch.no_grad():
        logits = model(**inputs).logits
    probs = torch.nn.functional.softmax(logits, dim=1).squeeze().tolist()
    return {id2label[str(i)]: round(probs[i], 3) for i in range(len(probs))}

In [50]:
from typing_extensions import Literal

MIN_DUR = 0.5         # turnos de menos de 0.5 s: muy poca voz para decidir, los saltamos
MAX_DUR = 15.0        # de los turnos largos usamos solo los primeros 15 s (basta y va más rápido)

def get_p(turno, gender:Literal["female", "male"]="female"):
    """Probabilidad de voz femenina de un turno, recortando el audio por sus tiempos de inicio y fin."""
    if turno.end - turno.start < MIN_DUR:
        return None
    trozo = audio[int(turno.start * SR): int(min(turno.end, turno.start + MAX_DUR) * SR)]
    return classify_audio(trozo)[gender]

df["p_female"] = df.apply(lambda row: get_p(row, "female"), axis=1)
df["p_male"] = df.apply(lambda row: get_p(row, "male"), axis=1)
df.dropna(subset=["p_female", "p_male"], inplace=True)

In [53]:
MIN_PROB = 0.80       # por debajo de esta probabilidad media, "no_determinado" (como en el script)
# Una etiqueta por hablante: media de sus turnos (más estable que decidir turno a turno)
p_hablante = df.groupby("speaker").p_female.mean()
p_hablante

speaker
Hablante_0    0.10170
Hablante_1    0.00100
Hablante_2    0.00100
Hablante_3    0.99175
Name: p_female, dtype: float64

In [54]:
genero = p_hablante.map(lambda p: "F" if p >= MIN_PROB else "M" if p <= 1 - MIN_PROB else "Unknown")
df["genero_inferido"] = df.speaker.map(genero).fillna("Unknown")
df[["start", "end", "speaker", "p_male","p_female", "genero_inferido"]].round(2)

,start,end,speaker,p_male,p_female,genero_inferido
0,0.00,1.60,Hablante_0,0.07,0.93,M
1,1.92,3.44,Hablante_0,0.96,0.04,M
2,3.68,7.10,Hablante_0,1.00,0.00,M
3,7.90,9.20,Hablante_0,0.96,0.04,M
4,10.04,18.98,Hablante_1,1.00,0.00,M
5,18.98,34.26,Hablante_1,1.00,0.00,M
6,34.92,37.28,Hablante_0,1.00,0.00,M
7,37.90,38.52,Hablante_0,1.00,0.00,M
8,39.74,40.36,Hablante_0,1.00,0.00,M
9,40.74,41.88,Hablante_0,1.00,0.00,M


## 5. Exportar el CSV del contrato de datos

Columnas acordadas con el equipo: `tiempo_i`, `tiempo_f`, `transcripcion`, `genero_inferido`.

In [63]:
contrato = (df.rename(columns={"start": "tiempo_i", "end": "tiempo_f", "text": "transcripcion"})
              [["tiempo_i", "tiempo_f", "speaker","transcripcion", "genero_inferido"]].round(2))
contrato.to_csv(Path(AUDIO).with_suffix(".csv").name, index=False, encoding="utf-8")
contrato.head()

,tiempo_i,tiempo_f,speaker,transcripcion,genero_inferido
0,0.00,1.60,Hablante_0,"Hey Pepe, mírame.",M
1,1.92,3.44,Hablante_0,¿De qué color es mi cara?,M
2,3.68,7.10,Hablante_0,El color... ¡verde!,M
3,7.90,9.20,Hablante_0,"Muy bien, Pepe.",M
4,10.04,18.98,Hablante_1,En España hay unos 100.000 niños y niñas con t...,M


## 6. Tiempo de habla por hablante

In [38]:
dur = (df.end - df.start).clip(lower=0)
tiempos = dur.groupby(df.speaker).sum().sort_values(ascending=False)

resumen = pd.DataFrame({"segundos": tiempos.round(1), "%": (100 * tiempos / tiempos.sum()).round(1)})
print(f"Total voz detectada: {tiempos.sum():.1f} s")
resumen

Total voz detectada: 127.1 s


,segundos,%
speaker,,
Hablante_2,52.5,41.3
Hablante_3,33.0,25.9
Hablante_1,24.2,19.1
Hablante_0,17.5,13.7


# 7. Tiempo de habla por género

In [62]:
df["duracion"] = (df.end - df.start).round(2)
agregados = df.groupby("genero_inferido").agg({"duracion": ["sum","count"]})
agregados[("duracion", "%")] = (100 * agregados[("duracion", "sum")] / agregados[("duracion", "sum")].sum())
agregados

duracion                 
                     sum count          %
genero_inferido                          
F                  32.74     4  25.803909
M                  94.14    17  74.196091